In [ ]:
import numpy as np
import nibabel as nib
from nilearn import plotting, datasets
from palettable.colorbrewer.diverging import RdBu_7
from matplotlib.colors import ListedColormap


In [ ]:
import numpy as np
import nibabel as nib
import scipy.io as sio
import matplotlib.pyplot as plt
from nilearn import plotting, datasets
from palettable.colorbrewer.diverging import RdBu_7

# Load same .mat files the main analysis uses
data_low  = sio.loadmat('outputs/NEW_4_factor_clustering_ipnybV4/low_anhedonia/results_Ceff_low_anhedonia.mat')
data_high = sio.loadmat('outputs/NEW_4_factor_clustering_ipnybV4/high_anhedonia/results_Ceff_high_anhedonia.mat')

hl_LOW  = data_low['hierarchicallevels_LOW']     # (NSUB_LOW, 232)
hl_HIGH = data_high['hierarchicallevels_HIGH']   # (NSUB_HIGH, 232)

# Subcortical = first 32 columns; mean across subjects -> length 32
subcort_LOW  = hl_LOW[:,  :32].mean(axis=0)
subcort_HIGH = hl_HIGH[:, :32].mean(axis=0)
subcort_DIFF = subcort_LOW - subcort_HIGH

# Cortical means too (used to set a shared color scale matching the cortical figure)
cort_LOW  = hl_LOW[:,  32:].mean(axis=0)
cort_HIGH = hl_HIGH[:, 32:].mean(axis=0)
cort_DIFF = cort_LOW - cort_HIGH

print('subcort_LOW :', subcort_LOW.shape, 'range', subcort_LOW.min(),  subcort_LOW.max())
print('subcort_HIGH:', subcort_HIGH.shape,'range', subcort_HIGH.min(), subcort_HIGH.max())
print('subcort_DIFF:', subcort_DIFF.shape,'range', subcort_DIFF.min(), subcort_DIFF.max())


In [ ]:
from PIL import Image

ATLAS_PATH = '/Users/proghani/Documents/personal/code_experiments/neuro/phd_stuff/tcp_parcellations/data/atlases/Tian_S2_subcortex.nii.gz'

_atlas_img = nib.load(ATLAS_PATH)
_atlas_labels = _atlas_img.get_fdata().astype(int)
_label_ids = np.arange(1, 33)   # Tian S2 IDs 1..32 (RH 1-16, LH 17-32)

def render_subcortex(values_32, vmin=None, vmax=None, out_path=None, dpi=600,
                    #  cut_coords=(-24, -12, -6, 6, 12, 24),
                     cut_coords=(-24, -12, 12, 24),
                     display_mode='x',
                     black_bg=False,
                     title=None):
    """
    Render 32 Tian-S2 subcortical values on top of the MNI152 brain template.

    If vmin/vmax are None, uses min/max of the data (matches MATLAB default).
    Draws anatomy first (plot_anat) so the brain is visible behind the parcels.

    NOTE: to squeeze the whitespace between slices, pass title=None here and put
    the label in the figure caption, then run tighten_montage() on the saved PNG.
    A baked-in title spans the inter-slice gaps and gets sliced when they're cropped.
    """
    assert len(values_32) == 32, f'expected 32 values, got {len(values_32)}'
    if vmin is None: vmin = float(np.min(values_32))
    if vmax is None: vmax = float(np.max(values_32))

    # Build value volume: NaN outside parcels, scalar value inside each parcel
    vol = np.full(_atlas_labels.shape, np.nan, dtype=float)
    for lid, v in zip(_label_ids, values_32):
        vol[_atlas_labels == lid] = v
    value_img = nib.Nifti1Image(vol, _atlas_img.affine, _atlas_img.header)

    # Colormap matched to MATLAB othercolor('RdBu7')
    cmap = RdBu_7.mpl_colormap.copy()

    bg = datasets.load_mni152_template(resolution=1)
    bg_data = bg.get_fdata().copy()
    bg_data[bg_data == 0] = bg_data.max()   # outside brain -> white
    bg = nib.Nifti1Image(bg_data, bg.affine, bg.header)

    # Draw anatomy first — full-slice field of view
    display = plotting.plot_anat(
        bg,
        display_mode=display_mode,
        cut_coords=list(cut_coords),
        annotate=False,
        draw_cross=False,
        black_bg=black_bg,
        dim=-0.25,
        title=title,
        colorbar=False,
    )
    # Overlay colored subcortical parcels on top
    display.add_overlay(value_img, cmap=cmap, vmin=vmin, vmax=vmax, colorbar=True)

    fig = display.frame_axes.figure
    fig.set_facecolor('white')
    display.frame_axes.set_facecolor('white')
    for ax in fig.axes:
        ax.set_facecolor('white')

    if out_path:
        display.savefig(out_path, dpi=dpi)
        print('saved', out_path)
    return display


def tighten_montage(in_png, out_path, gap_px=12, edge_px=10,
                    white=236, row_frac=0.98, band=(0.02, 0.98)):
    """
    Collapse the fully-white columns of a nilearn sagittal montage PNG so the
    slices sit closer together. nilearn's slice axes already tile edge-to-edge
    (an axes_locator overrides set_position), so the visible 'gap' is empty
    field-of-view *inside* each panel — the only reliable fix is to crop the
    white columns from the rendered image.

    gap_px   : white columns kept between brains  (lower = tighter; 0 ~ touching)
    edge_px  : outer left/right margin to keep
    white    : grayscale threshold above which a pixel counts as 'white'
    row_frac : fraction of band rows that must be white for a column to be cropped
    band     : (top, bottom) row fractions used to judge whiteness — keep away
               from any title/label strips so their columns aren't cropped
    """
    im = np.asarray(Image.open(in_png).convert('RGB'))
    H = im.shape[0]
    g = im[int(H * band[0]):int(H * band[1])].mean(axis=2)
    colwhite = (g > white).mean(axis=0) >= row_frac

    keep = np.ones(im.shape[1], bool)
    W = len(colwhite); i = 0
    while i < W:
        if colwhite[i]:
            j = i
            while j < W and colwhite[j]:
                j += 1
            target = edge_px if (i == 0 or j == W) else gap_px
            if j - i > target:
                keep[i:i + (j - i - target)] = False   # drop the excess white cols
            i = j
        else:
            i += 1

    Image.fromarray(im[:, keep, :]).save(out_path)
    print(f'tightened {im.shape[1]} -> {int(keep.sum())} px  ->  {out_path}')


In [ ]:
import os
os.makedirs('outputs/brain_rendering', exist_ok=True)

# LOW — own scale
render_subcortex(subcort_LOW,
                 out_path='outputs/brain_rendering/subcortical_hl_low.png',
                 title=None, dpi=600)
tighten_montage('outputs/brain_rendering/subcortical_hl_low.png',
                'outputs/brain_rendering/subcortical_hl_low_tight.png',
                gap_px=24, edge_px=20)   # 2x px because 600 dpi
plt.show()

# HIGH — own scale
render_subcortex(subcort_HIGH,
                 out_path='outputs/brain_rendering/subcortical_hl_high.png',
                 title=None, dpi=600)
tighten_montage('outputs/brain_rendering/subcortical_hl_high.png',
                'outputs/brain_rendering/subcortical_hl_high_tight.png',
                gap_px=24, edge_px=20)
plt.show()

# DIFF — symmetric scale around 0 (diverging colormap)
vabs = float(np.max(np.abs(subcort_DIFF)))
render_subcortex(subcort_DIFF, vmin=-vabs, vmax=vabs,
                 out_path='outputs/brain_rendering/subcortical_hl_diff.png',
                 title=None, dpi=600)
tighten_montage('outputs/brain_rendering/subcortical_hl_diff.png',
                'outputs/brain_rendering/subcortical_hl_diff_tight.png',
                gap_px=24, edge_px=20)
plt.show()
